## Save to Google Drive
Mounts your Drive. Every file is written atomically locally and to `MyDrive/phase107_testB`, and verified.

In [6]:
import os, json, datetime, hashlib
OUTDIR = '/content/drive/MyDrive/phase107_testB'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(OUTDIR, exist_ok=True)
except Exception as e:                      # not in Colab, or mount refused
    print('Drive not mounted:', e)
    OUTDIR = None
LOCALDIR = 'phase107_testB_out'
os.makedirs(LOCALDIR, exist_ok=True)


def _write(path, data, binary=False):
    """Atomic write: temp file, then replace; returns True only if the file reads back identical."""
    tmp = path + '.tmp'
    with open(tmp, 'wb' if binary else 'w') as f:
        f.write(data)
    os.replace(tmp, path)
    with open(path, 'rb' if binary else 'r') as f:
        return f.read() == data


def save(name, obj=None, raw=None, required=False):
    """Save JSON (obj) or bytes (raw) locally and to Drive. If required and neither copy succeeds, raise."""
    binary = raw is not None
    data = raw if binary else json.dumps(obj, indent=1, default=float)
    ok = []
    for d in [LOCALDIR] + ([OUTDIR] if OUTDIR else []):
        try:
            if _write(os.path.join(d, name), data, binary):
                ok.append('Drive' if d == OUTDIR else 'local')
        except Exception as e:
            print(f'WARNING: write to {d} failed: {e}')
    print(f'saved {name} ({" + ".join(ok) or "NOWHERE"})')
    if required and not ok:
        raise RuntimeError(f'could not save {name} anywhere; record it by hand: {data if not binary else "(binary)"}')
    return ok


def load(name, binary=False):
    for d in ([OUTDIR] if OUTDIR else []) + [LOCALDIR]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            with open(p, 'rb' if binary else 'r') as f:
                return f.read() if binary else json.load(f)
    return None

print('local:', LOCALDIR, '| Drive:', OUTDIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
local: phase107_testB_out | Drive: /content/drive/MyDrive/phase107_testB


# Phase 107 Test B: hardware target swap (IBM)

**What it tests.** In Phase 106's [[6,2]] code, one 4-qubit piece is predicted (exactly, from the stabilizer structure) to hold logical B's quantum information but only a classical trace of logical A. Each logical is entangled with a reference qubit (8 qubits in all). For each region and target we measure reference–region correlators and form the witness **E = c_X + c_Z**. Its two partner Paulis anticommute on the region (checked algebraically), so **E > 1 certifies entanglement between the reference and the region**. Ideal values are 2 where the target is recoverable and 1 where only a classical trace remains.

**What a pass shows.** That the hardware retains reference–region entanglement for B on the main piece and for A on the control piece, and that the main piece's witness is clearly larger for B than for A. Full recoverability versus non-recoverability is the exact theoretical prediction; the hardware tests the entanglement asymmetry it implies.

**Pass criteria** (frozen in `phase107/TESTB_ADDENDUM.md`; raw counts primary):
- **B1:** E_B(main) > 1 by ≥ 3 standard errors.
- **B2:** E_A(both) > 1 by ≥ 3 standard errors.
- **B3:** E_B(main) − E_A(main) > 0 by ≥ 3 standard errors.

The z-scores use a normal approximation, which is adequate for 4,000 shots with expectations away from ±1. They are not exact p-values.

**Order of use (first run).** All code cells in order. Each submission asks you to confirm the remaining QPU time first.

**Recovery after a disconnect.** Run the first five code cells (Drive, installs, plan, helpers, connection), then **only** the last cell. It reloads the saved manifest and compiled circuits; it never recompiles or resubmits. Submission cells check IBM itself, through job tags, for an existing Test B job before submitting, so a lost local file cannot trigger a second run.

**Regions** (from `testB_plan.json`): main = [2,3,6,7], both = [2,3,4,5], neither = [3,4,5,7].

In [7]:
# Runtime >= 0.40 is needed for the ibm_quantum_platform channel. After this cell: Runtime > Restart session, then run from the top.
!pip -q install -U "qiskit-ibm-runtime>=0.40" "qiskit>=2.0" "qiskit-aer>=0.17"

In [8]:
PLAN = {'adj': [90, 165, 66, 129, 33, 146, 5, 42],
 'edges': [[0, 1], [0, 3], [0, 4], [0, 6], [1, 2], [1, 5], [1, 7], [2, 6], [3, 7], [4, 5], [5, 7]],
 'transpile_seed': 1,
 'regions': {'main_B_not_A': [2, 3, 6, 7], 'both': [2, 3, 4, 5], 'neither': [3, 4, 5, 7]},
 'correlators': {'main_B_not_A/A': {'X': {'sign': 1, 'pauli': 'XIXZIIII', 'inside_region': False},
                                    'Z': {'sign': 1, 'pauli': 'ZIIXIIIZ', 'inside_region': True},
                                    'Y': {'sign': 1, 'pauli': 'YIIYIIZI', 'inside_region': False}},
                 'main_B_not_A/B': {'Z': {'sign': 1, 'pauli': 'IZXIIIZI', 'inside_region': True},
                                    'X': {'sign': 1, 'pauli': 'IXYYIIZX', 'inside_region': True},
                                    'Y': {'sign': -1, 'pauli': 'IYZYIIIX', 'inside_region': True}},
                 'both/A': {'X': {'sign': 1, 'pauli': 'XIXZZIII', 'inside_region': True},
                            'Z': {'sign': 1, 'pauli': 'ZIIIXZII', 'inside_region': True},
                            'Y': {'sign': 1, 'pauli': 'YIXZYZII', 'inside_region': True}},
                 'both/B': {'Z': {'sign': 1, 'pauli': 'IZIXYYII', 'inside_region': True},
                            'X': {'sign': 1, 'pauli': 'IXZXIZII', 'inside_region': True},
                            'Y': {'sign': -1, 'pauli': 'IYZIYXII', 'inside_region': True}},
                 'neither/A': {'X': {'sign': 1, 'pauli': 'XIIZZIII', 'inside_region': False},
                               'Z': {'sign': 1, 'pauli': 'ZIIXIIIZ', 'inside_region': True},
                               'Y': {'sign': 1, 'pauli': 'YIIYIXII', 'inside_region': False}},
                 'neither/B': {'Z': {'sign': 1, 'pauli': 'IZIIZXIZ', 'inside_region': True},
                               'X': {'sign': 1, 'pauli': 'IXIXIZII', 'inside_region': False},
                               'Y': {'sign': -1, 'pauli': 'IYIIYXII', 'inside_region': False}}}}

V, SHOTS = 8, 4000
PILOT_SHOTS = 1000

In [9]:
import math, getpass, io
import numpy as np
import qiskit, qiskit_aer, qiskit_ibm_runtime
from qiskit import QuantumCircuit, transpile, qpy
VERSIONS = {'qiskit': qiskit.__version__, 'qiskit_aer': qiskit_aer.__version__,
            'qiskit_ibm_runtime': qiskit_ibm_runtime.__version__}
print(VERSIONS)
TAG_PILOT, TAG_FULL = 'phase107-testB-pilot', 'phase107-testB-full'


def prep():
    qc = QuantumCircuit(V)
    qc.h(range(V))
    for u, v in PLAN['edges']:
        qc.cz(u, v)
    return qc


def settings():
    """(name, pauli string over virtual qubits, ideal sign, full partner inside region)."""
    out = []
    for key, cor in PLAN['correlators'].items():
        for L in ('X', 'Y', 'Z'):
            c = cor[L]
            out.append((f'{key}/{L}', c['pauli'], c['sign'], c['inside_region']))
    return out


def measure_circuit(pauli):
    qc = prep()
    for v, L in enumerate(pauli):
        if L == 'X':
            qc.h(v)
        elif L == 'Y':
            qc.sdg(v)
            qc.h(v)
    qc.measure_all()
    return qc


def calib_circuits():
    c0 = QuantumCircuit(V)
    c0.measure_all()
    c1 = QuantumCircuit(V)
    c1.x(range(V))
    c1.measure_all()
    return [c0, c1]


def bits_of(key):
    b = key.replace(' ', '')[::-1]            # b[v] = bit of virtual qubit v
    return [int(b[v]) for v in range(V)]


def readout_model(c0, c1):
    """Per qubit: p01 = P(read 1 | prepared 0), p10 = P(read 0 | prepared 1).
    For m = +1 (bit 0) / -1 (bit 1): E[m | z] = a z + b with a = 1 - p01 - p10, b = p10 - p01.
    The all-ones calibration also contains X-gate errors, which this model attributes to readout."""
    n0, n1 = sum(c0.values()), sum(c1.values())
    p01 = [sum(n for k, n in c0.items() if bits_of(k)[v] == 1) / n0 for v in range(V)]
    p10 = [sum(n for k, n in c1.items() if bits_of(k)[v] == 0) / n1 for v in range(V)]
    return [1 - p01[v] - p10[v] for v in range(V)], [p10[v] - p01[v] for v in range(V)]


def parity_stats(counts, support, a=None, b=None):
    """Mean and standard error of the parity over shots. Raw: product of m_i = +-1, SE from p~ = (k + 1) / (n + 2).
    Corrected: per-shot product of (m_i - b_i) / a_i; SE here is shot noise only (calibration noise via bootstrap)."""
    n = sum(counts.values())
    if a is None:
        k = sum(c for key, c in counts.items() if sum(bits_of(key)[v] for v in support) % 2 == 0)
        pt = (k + 1) / (n + 2)
        return (2 * k - n) / n, 2 * math.sqrt(pt * (1 - pt) / n)
    vals, wts = [], []
    for key, c in counts.items():
        bt = bits_of(key)
        x = 1.0
        for v in support:
            x *= ((1 - 2 * bt[v]) - b[v]) / a[v]
        vals.append(x)
        wts.append(c)
    vals, wts = np.array(vals), np.array(wts, dtype=float)
    mean = float(np.sum(vals * wts) / n)
    return mean, math.sqrt(float(np.sum(wts * (vals - mean) ** 2) / (n - 1)) / n)


def analyse(counts_list, sets, n_boot=0, seed=0):
    """counts_list: one dict per entry of sets, then the two calibration circuits (all 0, all 1).
    With n_boot > 0, corrected-E standard errors come from a bootstrap that resamples every count set, calibrations
    included."""
    a, b = readout_model(counts_list[-2], counts_list[-1])
    vals = {}
    for (name, pauli, sign, inside), counts in zip(sets, counts_list[:-2]):
        support = [v for v, L in enumerate(pauli) if L != 'I']
        r, rse = parity_stats(counts, support)
        c, cse = parity_stats(counts, support, a, b)
        vals[name] = {'raw': sign * r, 'raw_se': rse, 'corrected': sign * c, 'corrected_se': cse,
                      'ideal': 1.0 if inside else 0.0}
    keys = sorted({n.rsplit('/', 1)[0] for n, *_ in sets})
    stats = {}
    for key in keys:
        for kind in ('raw', 'corrected'):
            have = {L: vals.get(f'{key}/{L}') for L in 'XYZ'}
            if have['X'] is None or have['Z'] is None:
                continue
            E = have['X'][kind] + have['Z'][kind]
            se = math.sqrt(have['X'][kind + '_se'] ** 2 + have['Z'][kind + '_se'] ** 2)
            stats.setdefault(key, {})[kind] = {'E': E, 'E_se': se}
    if n_boot:
        rng = np.random.default_rng(seed)

        def resample(cnt):
            ks = list(cnt)
            p = np.array([cnt[k] for k in ks], dtype=float)
            draw = rng.multinomial(int(p.sum()), p / p.sum())
            return {k: int(d) for k, d in zip(ks, draw) if d}
        boot = {k: [] for k in stats}
        for _ in range(n_boot):
            cl = [resample(c) for c in counts_list]
            ab, bb = readout_model(cl[-2], cl[-1])
            for key in stats:
                e = 0.0
                for L in 'XZ':
                    i = [j for j, s in enumerate(sets) if s[0] == f'{key}/{L}'][0]
                    support = [v for v, LL in enumerate(sets[i][1]) if LL != 'I']
                    e += sets[i][2] * parity_stats(cl[i], support, ab, bb)[0]
                boot[key].append(e)
        for key in stats:
            stats[key]['corrected']['E_se'] = float(np.std(boot[key], ddof=1))
    tests = {}
    for kind in ('raw', 'corrected'):
        t = {}
        if 'main_B_not_A/B' in stats:
            eb = stats['main_B_not_A/B'][kind]
            t['B1'] = (eb['E'], (eb['E'] - 1) / eb['E_se'])
        if 'both/A' in stats:
            eab = stats['both/A'][kind]
            t['B2'] = (eab['E'], (eab['E'] - 1) / eab['E_se'])
        if 'main_B_not_A/B' in stats and 'main_B_not_A/A' in stats:
            eb, ea = stats['main_B_not_A/B'][kind], stats['main_B_not_A/A'][kind]
            t['B3'] = (eb['E'] - ea['E'], (eb['E'] - ea['E']) / math.sqrt(eb['E_se'] ** 2 + ea['E_se'] ** 2))
        tests[kind] = {k: {'value': v, 'z': z, 'holds': z > 3} for k, (v, z) in t.items()}
        if len(t) == 3:
            tests[kind]['pass'] = all(tests[kind][k]['holds'] for k in ('B1', 'B2', 'B3'))
    return {'correlators': vals, 'readout_a': a, 'readout_b': b, 'stats': stats, 'tests': tests,
            'bootstrap_resamples': n_boot}


def show(out, label):
    print(f'--- {label}')
    for key in sorted(out['stats']):
        st = out['stats'][key]
        print(f"  {key:16s} E raw {st['raw']['E']:+.3f} (se {st['raw']['E_se']:.3f})  "
              f"corrected {st['corrected']['E']:+.3f} (se {st['corrected']['E_se']:.3f})")
    for kind, t in out['tests'].items():
        line = '  '.join(f"{k} {t[k]['value']:+.3f} z={t[k]['z']:.1f} {'ok' if t[k]['holds'] else 'FAIL'}"
                         for k in ('B1', 'B2', 'B3') if k in t)
        verdict = '' if 'pass' not in t else ('  -> PASS' if t['pass'] else '  -> FAIL')
        print(f"  {kind:9s} {line}{verdict}")


SETS = settings()
NAMES = [s[0] for s in SETS] + ['calib/0', 'calib/1']
CIRCS = [measure_circuit(p) for _, p, _, _ in SETS] + calib_circuits()
print(len(CIRCS), 'circuits')

{'qiskit': '2.6.1', 'qiskit_aer': '0.18.0', 'qiskit_ibm_runtime': '0.51.0'}
20 circuits


## Connect to IBM Quantum
Uses the `IBM_QUANTUM_TOKEN` environment variable if set, otherwise asks for your API key (not stored). Check that `INSTANCE` names the instance with your free minutes. The cell lists any Test B jobs IBM already holds.

In [10]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

CHANNEL = "ibm_cloud"           # works on old and new qiskit-ibm-runtime versions with a CRN instance
INSTANCE = "<REDACTED-IBM-INSTANCE>"  # your IBM Quantum instance CRN
BACKEND_NAME = "ibm_fez"        # change to another Heron backend if preferred

token = os.environ.get('IBM_QUANTUM_TOKEN') or getpass.getpass("IBM Quantum API key: ")
service = QiskitRuntimeService(channel=CHANNEL, token=token, instance=INSTANCE)
backend = service.backend(BACKEND_NAME)
print('using', backend.name, '| pending jobs:', backend.status().pending_jobs)


def tagged_jobs(tag):
    """Test B jobs already known to IBM under this tag (any status)."""
    try:
        return list(service.jobs(job_tags=[tag], limit=20))
    except Exception as e:
        raise RuntimeError(f'could not query IBM for existing {tag} jobs ({e}); do not submit until this works')


def confirm_budget(expected_seconds):
    """Show what IBM reports about usage and ask for an explicit yes before submitting."""
    try:
        print('IBM usage report:', service.usage())
    except Exception as e:
        print('usage report unavailable here:', e, '- check the remaining time on the IBM Quantum dashboard')
    ans = input(f'This submission may use up to about {expected_seconds} QPU seconds. Type YES if your remaining '
                f'allowance covers it with margin: ')
    if ans.strip() != 'YES':
        raise RuntimeError('not confirmed; nothing submitted')

print('existing pilot jobs:', [j.job_id() for j in tagged_jobs(TAG_PILOT)])
print('existing full jobs :', [j.job_id() for j in tagged_jobs(TAG_FULL)])

IBM Quantum API key: ··········


qiskit_runtime_service._discover_account:WARNING:2026-10-23 17:39:59,087: Loading account with the given token. A saved account will not be used.


using ibm_fez | pending jobs: 0
existing pilot jobs: []
existing full jobs : []


## Compile for this backend (or reuse the saved compilation)
Optimization level 3, the plan's transpile seed and the state-preparation layout for every circuit. If anything was already submitted, the saved circuits are reloaded instead.

In [11]:
# Compile, or reuse the saved compilation if anything has already been submitted (provenance stays fixed).
saved_manifest = load('testB_manifest.json')
if saved_manifest and (load('testB_pilot_jobid.json') or load('testB_full_jobid.json') or tagged_jobs(TAG_PILOT)
                       or tagged_jobs(TAG_FULL)):
    raw = load('testB_transpiled.qpy', binary=True)
    assert raw is not None and hashlib.sha256(raw).hexdigest() == saved_manifest['qpy_sha256'], 'saved circuits missing or altered'
    TC = qpy.load(io.BytesIO(raw))
    LAYOUT, n2 = saved_manifest['layout'], saved_manifest['two_qubit_gates']
    assert saved_manifest['circuit_names'] == NAMES
    print('reusing the saved compilation from', saved_manifest['created_utc'])
else:
    base = prep()
    t0 = transpile(base, backend, optimization_level=3, seed_transpiler=PLAN['transpile_seed'])
    LAYOUT = [int(q) for q in t0.layout.initial_index_layout(filter_ancillas=True)]
    assert len(LAYOUT) == V and len(set(LAYOUT)) == V, LAYOUT
    TC = transpile(CIRCS, backend, optimization_level=3, initial_layout=LAYOUT,
                   seed_transpiler=PLAN['transpile_seed'])
    n2 = [sum(1 for ins in c.data if ins.operation.num_qubits == 2) for c in TC]
print('physical layout', LAYOUT)
print('two-qubit gates per circuit:', sorted(set(n2)), '| max depth', max(c.depth() for c in TC))
# every instruction, measurements included, must be supported on its physical qubits with its parameters
for c in TC:
    for ins in c.data:
        if ins.operation.name == 'barrier':
            continue
        qs = tuple(c.find_bit(q).index for q in ins.qubits)
        assert backend.target.instruction_supported(ins.operation.name, qs, parameters=list(ins.operation.params)), \
            (ins.operation.name, qs, ins.operation.params)
print('all instructions supported by', backend.name)

physical layout [131, 130, 129, 132, 138, 151, 118, 133]
two-qubit gates per circuit: [0, 26] | max depth 27
all instructions supported by ibm_fez


## Noise-free check (no QPU time; must pass)

In [12]:
# Noise-free check of every compiled circuit against the exact expected values (no QPU time).
from qiskit_aer import AerSimulator
ideal = AerSimulator().run(TC, shots=SHOTS, seed_simulator=1071).result()
bad = []
for i, (name, pauli, sign, inside) in enumerate(SETS):
    support = [v for v, L in enumerate(pauli) if L != 'I']
    r, _ = parity_stats(ideal.get_counts(i), support)
    want = sign * (1.0 if inside else 0.0)       # truncated operators: expectation 0 (tolerance ~5 SE at 4,000 shots)
    if abs(r - want) > (1e-9 if inside else 0.08):
        bad.append((name, r, want))
c0, c1 = ideal.get_counts(len(SETS)), ideal.get_counts(len(SETS) + 1)
assert c0.get('0' * V, 0) == SHOTS and c1.get('1' * V, 0) == SHOTS, 'calibration circuits wrong'
assert not bad, bad
print('noise-free check passed for all', len(SETS), 'correlator circuits and both calibrations')

noise-free check passed for all 18 correlator circuits and both calibrations


## Optional noisy simulation with today's calibration (no QPU time)

In [13]:
# Optional: noisy simulation with this backend's current noise model (no QPU time).
sim = AerSimulator.from_backend(backend)
res = sim.run(TC, shots=SHOTS, seed_simulator=1072).result()
sim_out = analyse([res.get_counts(i) for i in range(len(TC))], SETS, n_boot=200)
show(sim_out, f'Aer with {backend.name} noise model')

--- Aer with ibm_fez noise model
  both/A           E raw +1.812 (se 0.009)  corrected +2.169 (se 0.019)
  both/B           E raw +1.753 (se 0.011)  corrected +1.873 (se 0.012)
  main_B_not_A/A   E raw +0.780 (se 0.018)  corrected +0.914 (se 0.023)
  main_B_not_A/B   E raw +1.651 (se 0.013)  corrected +1.785 (se 0.016)
  neither/A        E raw +0.777 (se 0.018)  corrected +0.911 (se 0.023)
  neither/B        E raw +0.822 (se 0.018)  corrected +0.871 (se 0.018)
  raw       B1 +1.651 z=52.1 ok  B2 +1.812 z=86.0 ok  B3 +0.871 z=39.2 ok  -> PASS
  corrected B1 +1.785 z=50.0 ok  B2 +2.169 z=60.5 ok  B3 +0.871 z=31.4 ok  -> PASS


## Manifest (saved once, before any submission)

In [14]:
# Save the manifest and compiled circuits ONCE, before any submission. Never overwritten afterwards.
if load('testB_manifest.json'):
    MANIFEST_DATA = load('testB_manifest.json')
    print('manifest already saved at', MANIFEST_DATA['created_utc'], '- kept unchanged')
else:
    buf = io.BytesIO()
    qpy.dump(TC, buf)
    QPY_BYTES = buf.getvalue()
    MANIFEST_DATA = {'created_utc': datetime.datetime.utcnow().isoformat(), 'backend': backend.name,
                     'versions': VERSIONS, 'layout': LAYOUT, 'two_qubit_gates': n2, 'shots': SHOTS,
                     'pilot_shots': PILOT_SHOTS, 'circuit_names': NAMES,
                     'qpy_sha256': hashlib.sha256(QPY_BYTES).hexdigest(), 'plan': PLAN}
    save('testB_transpiled.qpy', raw=QPY_BYTES, required=True)
    save('testB_manifest.json', MANIFEST_DATA, required=True)

saved testB_transpiled.qpy (local + Drive)
saved testB_manifest.json (local + Drive)


/tmp/ipykernel_1016/1317199342.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  MANIFEST_DATA = {'created_utc': datetime.datetime.utcnow().isoformat(), 'backend': backend.name,


## Pilot (about 6,000 shots): a hardware-health check with an abort rule

In [16]:
# Pilot: a hardware-health check on the four circuits that enter B1 and B2, plus both calibrations (6 x 1,000 shots).
# Abort rule (fixed before running): no full run if either pilot witness is clearly below 1, i.e. E + 2 SE < 1.
# It is a health check, not a scientific test.
PILOT_NAMES = ['main_B_not_A/B/X', 'main_B_not_A/B/Z', 'both/A/X', 'both/A/Z']
idx = [i for i, s in enumerate(SETS) if s[0] in PILOT_NAMES]
pilot_tc = [TC[i] for i in idx] + [TC[-2], TC[-1]]
if load('testB_full_jobid.json') or tagged_jobs(TAG_FULL):
    raise RuntimeError('a full Test B job already exists; no pilot. Go to the last cell to retrieve it.')
existing = tagged_jobs(TAG_PILOT)
prev = load('testB_pilot_jobid.json')
if prev or existing:
    pid = prev['job_id'] if prev else existing[0].job_id()
    print('pilot already submitted:', pid, '(retrieving, not resubmitting)')
    pilot = service.job(pid)
else:
    confirm_budget(120)
    sampler = SamplerV2(mode=backend)
    sampler.options.max_execution_time = 120
    sampler.options.environment.job_tags = [TAG_PILOT]
    pilot = sampler.run(pilot_tc, shots=PILOT_SHOTS)
    print('PILOT JOB ID (write it down):', pilot.job_id())
    save('testB_pilot_jobid.json', {'job_id': pilot.job_id(), 'circuits': [NAMES[i] for i in idx] + ['calib/0', 'calib/1']},
         required=True)
pr = pilot.result()
pcounts = [pr[i].data.meas.get_counts() for i in range(len(pilot_tc))]
pilot_out = analyse(pcounts, [SETS[i] for i in idx])
show(pilot_out, 'PILOT')
try:
    pilot_usage = pilot.usage()
except Exception:
    pilot_usage = pilot.metrics().get('usage')
print('pilot usage:', pilot_usage)
save('testB_pilot_result.json', {'job_id': pilot.job_id(), 'usage': pilot_usage, 'counts': pcounts, 'analysis': pilot_out})
GO = all(pilot_out['stats'][k]['raw']['E'] + 2 * pilot_out['stats'][k]['raw']['E_se'] >= 1.0
         for k in ('main_B_not_A/B', 'both/A'))
print('GO for full run' if GO else 'ABORT: a pilot witness is clearly below 1; do not submit the full job')

IBM usage report: {'instance_id': '<REDACTED-IBM-INSTANCE>', 'plan_id': '850b21a7-71de-4e53-9441-1abdd202f35d', 'usage_consumed_seconds': 0, 'usage_consumed_ru': 0, 'usage_period': {'start_time': '2026-09-25T17:41:20.601Z', 'end_time': '2026-10-23T17:41:20.601Z'}, 'usage_limit_seconds': 600, 'usage_limit_ru': 10, 'usage_limit_reached': False, 'usage_remaining_seconds': 600}
This submission may use up to about 120 QPU seconds. Type YES if your remaining allowance covers it with margin: YES


/tmp/ipykernel_1016/220645382.py:17: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = SamplerV2(mode=backend)


PILOT JOB ID (write it down): dbdpohk24mrc739iq8sg
saved testB_pilot_jobid.json (local + Drive)
--- PILOT
  both/A           E raw +1.598 (se 0.027)  corrected +1.940 (se 0.032)
  main_B_not_A/B   E raw +1.454 (se 0.031)  corrected +1.545 (se 0.033)
  raw       B1 +1.454 z=14.8 ok  B2 +1.598 z=22.3 ok
  corrected B1 +1.545 z=16.7 ok  B2 +1.940 z=29.0 ok
pilot usage: 4
saved testB_pilot_result.json (local + Drive)
GO for full run


## Submit the full job (once)

In [17]:
# Submit the full job ONCE. IBM is checked for an existing tagged job first; the ID is saved before waiting.
assert GO, 'pilot abort rule triggered'
existing = tagged_jobs(TAG_FULL)
prev = load('testB_full_jobid.json')
if prev or existing:
    FULL_JOB_ID = prev['job_id'] if prev else existing[0].job_id()
    print('full job already submitted:', FULL_JOB_ID, '- run the last cell to retrieve it')
else:
    confirm_budget(300)
    sampler = SamplerV2(mode=backend)
    sampler.options.max_execution_time = 300
    sampler.options.environment.job_tags = [TAG_FULL]
    job = sampler.run(TC, shots=SHOTS)
    FULL_JOB_ID = job.job_id()
    print('FULL JOB ID (write it down):', FULL_JOB_ID)
    save('testB_full_jobid.json', {'job_id': FULL_JOB_ID, 'backend': backend.name,
                                   'submitted_utc': datetime.datetime.utcnow().isoformat()}, required=True)

IBM usage report: {'instance_id': '<REDACTED-IBM-INSTANCE>', 'plan_id': '850b21a7-71de-4e53-9441-1abdd202f35d', 'usage_consumed_seconds': 4, 'usage_consumed_ru': 0.054454, 'usage_period': {'start_time': '2026-09-25T17:42:30.746Z', 'end_time': '2026-10-23T17:42:30.746Z'}, 'usage_limit_seconds': 600, 'usage_limit_ru': 10, 'usage_limit_reached': False, 'usage_remaining_seconds': 596}
This submission may use up to about 300 QPU seconds. Type YES if your remaining allowance covers it with margin: YES


/tmp/ipykernel_1016/4150906341.py:10: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = SamplerV2(mode=backend)


FULL JOB ID (write it down): dbdpp41gv2cc73dotnv0
saved testB_full_jobid.json (local + Drive)


/tmp/ipykernel_1016/4150906341.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'submitted_utc': datetime.datetime.utcnow().isoformat()}, required=True)


## Retrieve, analyse and save (also the recovery cell)

In [18]:
# Retrieve. Safe to rerun, and the only cell needed after a disconnect (with cells 1-5).
# It uses the saved manifest and job ID (or IBM's tagged job), never recompiles and never submits.
MANIFEST_DATA = load('testB_manifest.json')
assert MANIFEST_DATA is not None, 'manifest missing: recover it from Drive before analysing'
raw = load('testB_transpiled.qpy', binary=True)
assert raw is not None and hashlib.sha256(raw).hexdigest() == MANIFEST_DATA['qpy_sha256'], 'saved circuits missing or altered'
assert MANIFEST_DATA['circuit_names'] == NAMES, 'circuit order differs from the analysis order'
rec = load('testB_full_jobid.json')
FULL_JOB_ID = rec['job_id'] if rec else tagged_jobs(TAG_FULL)[0].job_id()
job = service.job(FULL_JOB_ID)
result = job.result()
COUNTS = [result[i].data.meas.get_counts() for i in range(len(NAMES))]
save('testB_raw_counts.json', {'job_id': FULL_JOB_ID, 'backend': MANIFEST_DATA['backend'],
                               'circuit_names': NAMES, 'counts': COUNTS}, required=True)
try:
    usage = job.usage()
except Exception:
    usage = job.metrics().get('usage')
print('usage:', usage)
hw_out = analyse(COUNTS, SETS, n_boot=500)
show(hw_out, f"HARDWARE {MANIFEST_DATA['backend']}")
save('testB_hardware_result.json', {'job_id': FULL_JOB_ID, 'usage': usage, 'manifest': MANIFEST_DATA, 'analysis': hw_out})
try:
    from google.colab import files
    files.download(os.path.join(LOCALDIR, 'testB_hardware_result.json'))
except Exception:
    pass

saved testB_raw_counts.json (local + Drive)
usage: 23
--- HARDWARE ibm_fez
  both/A           E raw +1.550 (se 0.014)  corrected +1.859 (se 0.021)
  both/B           E raw +1.548 (se 0.014)  corrected +1.641 (se 0.016)
  main_B_not_A/A   E raw +0.812 (se 0.019)  corrected +0.956 (se 0.024)
  main_B_not_A/B   E raw +1.521 (se 0.014)  corrected +1.618 (se 0.017)
  neither/A        E raw +0.811 (se 0.018)  corrected +0.955 (se 0.021)
  neither/B        E raw +0.835 (se 0.018)  corrected +0.890 (se 0.019)
  raw       B1 +1.521 z=35.9 ok  B2 +1.550 z=39.2 ok  B3 +0.709 z=30.0 ok  -> PASS
  corrected B1 +1.618 z=37.4 ok  B2 +1.859 z=40.5 ok  B3 +0.663 z=22.8 ok  -> PASS
saved testB_hardware_result.json (local + Drive)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>